In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


In [ ]:
# ============================================================
# INITIALIZATION: PATHS AND FOLDERS
# ============================================================

from pathlib import Path
import os

# ============================================================
# TASK PARAMETER
# Change only this value: "sst" or "mnli"
# ============================================================
TASK = "sst"

if TASK not in ["sst", "mnli"]:
    raise ValueError("TASK must be either 'sst' or 'mnli'")

# ============================================================
# TASK CONFIGURATION
# ============================================================
TASK_CONFIG = {
    "sst": {
        "task_name": "SST",
        "dataset_subdir": "SST",
        "num_labels": 2,
        "model_dir_name": "sst_bert_base_uncased",
        "results_prefix": "sst",
    },
    "mnli": {
        "task_name": "MNLI",
        "dataset_subdir": "MNLI",
        "num_labels": 3,
        "model_dir_name": "mnli_bert_base_uncased",
        "results_prefix": "mnli",
    },
}

CONFIG = TASK_CONFIG[TASK]

TASK_NAME = CONFIG["task_name"]
DATASET_SUBDIR = CONFIG["dataset_subdir"]
NUM_LABELS = CONFIG["num_labels"]
MODEL_DIR_NAME = CONFIG["model_dir_name"]
RESULTS_PREFIX = CONFIG["results_prefix"]

# ------------------------------------------------------------
# Working directory
# ------------------------------------------------------------
WORK_DIR = Path(str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines/TrainingOfUncasedModel'))

# Output directories
RESULTS_DIR = WORK_DIR / "Results"
MODELS_DIR = WORK_DIR / "Models"

TASK_RESULTS_DIR = RESULTS_DIR / RESULTS_PREFIX
TASK_MODEL_DIR = MODELS_DIR / MODEL_DIR_NAME

# Dataset directory
DATASET_DIR = Path(str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Preprocessing/Dataset/ProcessedDB'))
TASK_DATASET_DIR = DATASET_DIR / DATASET_SUBDIR

# ------------------------------------------------------------
# Create output folders if they do not exist
# ------------------------------------------------------------
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TASK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TASK_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Check required directories
# ------------------------------------------------------------
required_dirs = {
    "WORK_DIR": WORK_DIR,
    "RESULTS_DIR": RESULTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "TASK_RESULTS_DIR": TASK_RESULTS_DIR,
    "TASK_MODEL_DIR": TASK_MODEL_DIR,
    "DATASET_DIR": DATASET_DIR,
    "TASK_DATASET_DIR": TASK_DATASET_DIR,
}

for name, path in required_dirs.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

# ------------------------------------------------------------
# Dataset files for selected task
# ------------------------------------------------------------
dataset_files = {
    "X_train": TASK_DATASET_DIR / "X_train.parquet",
    "X_val": TASK_DATASET_DIR / "X_val.parquet",
    "X_test": TASK_DATASET_DIR / "X_test.parquet",
    "y_train": TASK_DATASET_DIR / "y_train.parquet",
    "y_val": TASK_DATASET_DIR / "y_val.parquet",
    "y_test": TASK_DATASET_DIR / "y_test.parquet",
}

# ------------------------------------------------------------
# Check required files
# ------------------------------------------------------------
for split_name, file_path in dataset_files.items():
    if not file_path.exists():
        raise FileNotFoundError(f"{TASK_NAME} file missing: {split_name} -> {file_path}")

# ------------------------------------------------------------
# Set notebook working directory
# ------------------------------------------------------------
os.chdir(WORK_DIR)

print("Initialization completed successfully.")
print(f"Selected task: {TASK_NAME}")
print(f"Number of labels: {NUM_LABELS}")
print(f"Working directory: {WORK_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Task results directory: {TASK_RESULTS_DIR}")
print(f"Task model directory: {TASK_MODEL_DIR}")
print(f"Task dataset directory: {TASK_DATASET_DIR}")

In [ ]:
# ============================================================
# LOAD PROCESSED DATASET
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------
def load_parquet_split(files_dict):
    X_train = pd.read_parquet(files_dict["X_train"])
    X_val   = pd.read_parquet(files_dict["X_val"])
    X_test  = pd.read_parquet(files_dict["X_test"])

    y_train = pd.read_parquet(files_dict["y_train"])
    y_val   = pd.read_parquet(files_dict["y_val"])
    y_test  = pd.read_parquet(files_dict["y_test"])

    # Convert labels from DataFrame to Series if they have one column
    if isinstance(y_train, pd.DataFrame) and y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    if isinstance(y_val, pd.DataFrame) and y_val.shape[1] == 1:
        y_val = y_val.iloc[:, 0]
    if isinstance(y_test, pd.DataFrame) and y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]

    return X_train, X_val, X_test, y_train, y_val, y_test


# ------------------------------------------------------------
# Load selected task dataset
# ------------------------------------------------------------
X_train, X_val, X_test, y_train, y_val, y_test = load_parquet_split(dataset_files)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)


# ------------------------------------------------------------
# Print dataset information
# ------------------------------------------------------------
print(f"{TASK_NAME} loaded successfully")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print(f"{TASK_NAME} labels:", sorted(y_train.unique()))

print(f"\n{TASK_NAME} columns:", list(X_train.columns))

In [ ]:
# ============================================================
# DEBUGGING TO BE REMOVED: USE ONLY A FRACTION OF THE DATA
# ============================================================

from sklearn.model_selection import train_test_split

# Modify this value when needed
DATA_FRACTION = 1
RANDOM_STATE = 42


def reduce_dataset_fraction(X, y, fraction=0.10, random_state=42):
    """
    Keep only a fraction of the dataset.
    Stratification is used to preserve label distribution.
    """
    if fraction >= 1.0:
        return X.reset_index(drop=True), y.reset_index(drop=True)

    X_small, _, y_small, _ = train_test_split(
        X,
        y,
        train_size=fraction,
        random_state=random_state,
        stratify=y
    )

    return X_small.reset_index(drop=True), y_small.reset_index(drop=True)


# ------------------------------------------------------------
# Reduce selected task dataset
# ------------------------------------------------------------
X_train, y_train = reduce_dataset_fraction(
    X_train,
    y_train,
    DATA_FRACTION,
    RANDOM_STATE
)

X_val, y_val = reduce_dataset_fraction(
    X_val,
    y_val,
    DATA_FRACTION,
    RANDOM_STATE
)

X_test, y_test = reduce_dataset_fraction(
    X_test,
    y_test,
    DATA_FRACTION,
    RANDOM_STATE
)


print(f"Using {DATA_FRACTION * 100:.1f}% of the {TASK_NAME} dataset")

print(f"\n{TASK_NAME}")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# ============================================================
# LOAD BERT-BASE-UNCASED BACKBONE, TOKENIZER AND CUSTOM CLASSIFIER
# ============================================================

import sys
import torch
import numpy as np
import torch.nn as nn
import gc

from transformers import AutoTokenizer, AutoModel

# ============================================================
# PARAMETERS
# ============================================================
SEED = 42

# ============================================================
# PATHS
# ============================================================
BASELINES_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines')

# ============================================================
# MODEL ID
# ============================================================
MODEL_ID = "google-bert/bert-base-uncased"

# ============================================================
# IMPORT CUSTOM CLASSIFIER
# ============================================================
if BASELINES_PATH not in sys.path:
    sys.path.append(BASELINES_PATH)

from classifier import Classifier

# ============================================================
# SET RANDOM SEEDS
# ============================================================
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ============================================================
# LOAD BERT BACKBONE FROM HUGGING FACE
# AutoModel loads BERT without classification head
# ============================================================
bert_backbone = AutoModel.from_pretrained(MODEL_ID)

# ============================================================
# CREATE TASK-SPECIFIC CUSTOM CLASSIFIER
# NUM_LABELS is defined from TASK configuration
# ============================================================
classifier = Classifier(class_number=NUM_LABELS)

# ============================================================
# MOVE MODELS TO DEVICE
# ============================================================
bert_backbone = bert_backbone.to(device)
classifier = classifier.to(device)

# ============================================================
# KEEP EVERYTHING TRAINABLE
# ============================================================
bert_backbone.train()
classifier.train()

for param in bert_backbone.parameters():
    param.requires_grad = True

for param in classifier.parameters():
    param.requires_grad = True

# ============================================================
# CLEAN MEMORY
# ============================================================
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================
# PARAMETER COUNT
# ============================================================
bert_trainable_params = sum(
    p.numel() for p in bert_backbone.parameters() if p.requires_grad
)

classifier_trainable_params = sum(
    p.numel() for p in classifier.parameters() if p.requires_grad
)

trainable_params = bert_trainable_params + classifier_trainable_params

# ============================================================
# PRINT INFORMATION
# ============================================================
print("Loaded model:", MODEL_ID)
print("Selected task:", TASK_NAME)
print("Number of labels:", NUM_LABELS)
print("Tokenizer loaded.")
print("BERT backbone loaded without classification head.")
print("Custom classifier created.")
print("Backbone is trainable.")
print("Classifier is trainable.")
print("Device:", device)

print(f"BERT trainable parameters: {bert_trainable_params:,}")
print(f"Classifier trainable parameters: {classifier_trainable_params:,}")
print(f"Total trainable parameters: {trainable_params:,}")

# ============================================================
# PRINT MODEL ARCHITECTURES
# ============================================================
print("\n" + "=" * 80)
print("BERT BACKBONE ARCHITECTURE")
print("=" * 80)
print(bert_backbone)

print("\n" + "=" * 80)
print("CUSTOM CLASSIFIER ARCHITECTURE")
print("=" * 80)
print(classifier)

# ============================================================
# CHECK THAT BERT CLASSIFIER HEAD IS NOT PRESENT
# ============================================================
print("\nBackbone type:", type(bert_backbone))
print("Backbone has classifier:", hasattr(bert_backbone, "classifier"))

In [ ]:
# ============================================================
# CLEAN MEMORY AND CHECK TRAINABLE BERT BACKBONE
# ============================================================

import gc
import torch

# ------------------------------------------------------------
# Remove objects that are no longer needed from previous baselines
# ------------------------------------------------------------
objects_to_delete = [
    "sst_backbone",
    "mnli_backbone",
    "merged_backbone",
    "sst_state_dict",
    "mnli_state_dict",
    "avg_state_dict",
    "avg_backbone",
    "sst_model_full",
    "mnli_model_full",
]

for obj_name in objects_to_delete:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# Ensure backbone and classifier are on device and trainable
# ------------------------------------------------------------
bert_backbone = bert_backbone.to(device)
classifier = classifier.to(device)

bert_backbone.train()
classifier.train()

for param in bert_backbone.parameters():
    param.requires_grad = True

for param in classifier.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------
bert_total_params = sum(p.numel() for p in bert_backbone.parameters())
bert_trainable_params = sum(
    p.numel() for p in bert_backbone.parameters() if p.requires_grad
)

classifier_total_params = sum(p.numel() for p in classifier.parameters())
classifier_trainable_params = sum(
    p.numel() for p in classifier.parameters() if p.requires_grad
)

trainable_params = bert_trainable_params + classifier_trainable_params

print("Memory cleaned successfully.")
print("BERT backbone and classifier ready.")
print(f"Model ID: {MODEL_ID}")
print(f"Selected task: {TASK_NAME}")
print(f"BERT total parameters: {bert_total_params:,}")
print(f"BERT trainable parameters: {bert_trainable_params:,}")
print(f"Classifier total parameters: {classifier_total_params:,}")
print(f"Classifier trainable parameters: {classifier_trainable_params:,}")
print(f"Total trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

In [ ]:
# ============================================================
# CHECK CUSTOM CLASSIFIER
# ============================================================

classifier.train()

for param in classifier.parameters():
    param.requires_grad = True

classifier_trainable_params = sum(
    p.numel() for p in classifier.parameters() if p.requires_grad
)

print(f"{TASK_NAME} classifier ready.")
print(f"Number of labels: {NUM_LABELS}")
print(f"Classifier trainable parameters: {classifier_trainable_params:,}")

In [ ]:
# ============================================================
# TOKENIZATION AND DATALOADERS
# ============================================================

from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch
import pandas as pd

# ============================================================
# BATCH SIZE
# ============================================================
BATCH_SIZE = 32

# ============================================================
# Utility: convert X/y to clean lists
# ============================================================
def extract_texts(X):
    if isinstance(X, pd.DataFrame):
        if X.shape[1] != 1:
            raise ValueError(f"Expected X with 1 column, found {X.shape[1]}")
        return X.iloc[:, 0].astype(str).tolist()

    if isinstance(X, pd.Series):
        return X.astype(str).tolist()

    return pd.Series(X).astype(str).tolist()


def extract_labels(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y with 1 column, found {y.shape[1]}")
        return y.iloc[:, 0].astype(int).values

    if isinstance(y, pd.Series):
        return y.astype(int).values

    return pd.Series(y).astype(int).values


# ============================================================
# Utility: prepare inputs
# MNLI is sentence-pair classification
# SST is single-sentence classification
# ============================================================
def prepare_texts(task, X):
    texts = extract_texts(X)

    if task == "mnli":
        premises = []
        hypotheses = []

        for t in texts:
            parts = t.split("[SEP]", 1)
            premises.append(parts[0].strip())
            hypotheses.append(parts[1].strip() if len(parts) > 1 else "")

        return premises, hypotheses

    return texts, None


# ============================================================
# Inspect tokenized length distribution
# ============================================================
lengths = []

all_splits_for_length = [
    (TASK, X_train, "train"),
    (TASK, X_val,   "val"),
    (TASK, X_test,  "test"),
]

for task, X_split, split_name in all_splits_for_length:
    texts_a, texts_b = prepare_texts(task, X_split)

    for i in tqdm(range(len(texts_a)), desc=f"Inspecting lengths {TASK_NAME} {split_name}"):
        if texts_b is None:
            enc = tokenizer(texts_a[i], truncation=False)
        else:
            enc = tokenizer(texts_a[i], texts_b[i], truncation=False)

        lengths.append(len(enc["input_ids"]))

percentiles = [90, 95, 97, 99]

print("Tokenized length percentiles:")
for p in percentiles:
    print(f"{p}% :", int(np.percentile(lengths, p)))

# Match RoutedModel: fixed truncation cap, padding to the longest item in each split.
# The percentiles above are diagnostic only; they do not change this shared cap.
MAX_LEN = 256

print("Truncation cap (same as RoutedModel):", MAX_LEN)


# ============================================================
# Tokenization utility
# ============================================================
def tokenize_split(X, y, task, split_name):
    texts_a, texts_b = prepare_texts(task, X)
    labels_array = extract_labels(y)

    if len(texts_a) != len(labels_array):
        raise ValueError(f"Length mismatch in {split_name}: X={len(texts_a)}, y={len(labels_array)}")
    if not texts_a:
        raise ValueError(f"Cannot tokenize an empty split: {split_name}")

    # Tokenize in chunks without padding. Padding each chunk independently
    # would produce different widths and would not match RoutedModel.
    encoded_split = {"input_ids": [], "attention_mask": [], "token_type_ids": []}
    for i in tqdm(range(0, len(texts_a), 1024), desc=f"Tokenizing {split_name}"):
        batch_a = texts_a[i:i + 1024]
        batch_b = texts_b[i:i + 1024] if texts_b is not None else None
        enc = tokenizer(
            batch_a,
            text_pair=batch_b,
            padding=False,
            truncation=True,
            max_length=MAX_LEN,
            return_token_type_ids=True
        )
        encoded_split["input_ids"].extend(enc["input_ids"])
        encoded_split["attention_mask"].extend(enc["attention_mask"])
        encoded_split["token_type_ids"].extend(
            enc.get("token_type_ids", [[0] * len(ids) for ids in enc["input_ids"]])
        )

    # Pad once across the entire split, after truncation, exactly as the
    # routed tokenizer(texts, text_pair=..., padding=True, max_length=256).
    encodings = tokenizer.pad(encoded_split, padding=True, return_tensors="pt")
    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    token_type_ids = encodings["token_type_ids"]
    labels = torch.tensor(labels_array, dtype=torch.long)

    print(f"{split_name}: padded sequence length = {input_ids.shape[1]} (cap {MAX_LEN})")
    return TensorDataset(input_ids, attention_mask, token_type_ids, labels)


# ============================================================
# Build selected task datasets
# ============================================================
train_dataset = tokenize_split(X_train, y_train, TASK, "train")
val_dataset   = tokenize_split(X_val,   y_val,   TASK, "val")
test_dataset  = tokenize_split(X_test,  y_test,  TASK, "test")


# ============================================================
# Build dataloaders
# ============================================================
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)


print("Tokenization completed successfully.")
print("Selected task:", TASK_NAME)
print("MAX_LEN:", MAX_LEN)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# TRAINING MECHANISMS INITIALIZATION
# ============================================================

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ============================================================
# TRAINING PARAMETERS
# ============================================================
NUM_EPOCHS = 3

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4

LR_FACTOR = 0.1
LR_PATIENCE = 2
LR_THRESHOLD = 1e-4
MIN_LR = 1e-6

# ============================================================
# LOSS FUNCTION
# ============================================================
criterion = nn.CrossEntropyLoss()

# ============================================================
# OPTIMIZER
# Full fine-tuning: BERT backbone + custom classifier
# ============================================================
optimizer = AdamW(
    list(bert_backbone.parameters()) + list(classifier.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ============================================================
# SCHEDULER
# Reduce LR when validation loss stops improving
# ============================================================
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    threshold=LR_THRESHOLD,
    threshold_mode="abs",
    min_lr=MIN_LR
)

# ============================================================
# EARLY STOPPING VARIABLES
# ============================================================
best_val_f1 = -float("inf")
best_val_loss = float("inf")
best_epoch = -1
early_stopping_counter = 0

# ============================================================
# BEST MODEL PATH
# ============================================================
BEST_MODEL_DIR = TASK_MODEL_DIR / "best_checkpoint"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_BACKBONE_PATH = BEST_MODEL_DIR / "bert_backbone.pt"
BEST_CLASSIFIER_PATH = BEST_MODEL_DIR / "classifier.pt"
BEST_TRAINING_INFO_PATH = BEST_MODEL_DIR / "training_info.json"

# ============================================================
# TRAINING HISTORY
# ============================================================
training_history = []

# ============================================================
# CHECK TRAINABLE PARAMETERS
# ============================================================
bert_trainable_params = sum(
    p.numel() for p in bert_backbone.parameters() if p.requires_grad
)

classifier_trainable_params = sum(
    p.numel() for p in classifier.parameters() if p.requires_grad
)

trainable_params = bert_trainable_params + classifier_trainable_params

bert_total_params = sum(p.numel() for p in bert_backbone.parameters())
classifier_total_params = sum(p.numel() for p in classifier.parameters())

total_params = bert_total_params + classifier_total_params

print("Training mechanisms initialized.")
print(f"Selected task: {TASK_NAME}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Early stopping min delta: {EARLY_STOPPING_MIN_DELTA}")
print(f"LR factor: {LR_FACTOR}")
print(f"LR patience: {LR_PATIENCE}")
print(f"LR threshold: {LR_THRESHOLD}")
print(f"Minimum LR: {MIN_LR}")
print(f"BERT trainable parameters: {bert_trainable_params:,}")
print(f"Classifier trainable parameters: {classifier_trainable_params:,}")
print(f"Total trainable parameters: {trainable_params:,}")
print(f"Total model parameters: {total_params:,}")
print(f"Best model directory: {BEST_MODEL_DIR}")

In [ ]:
# ============================================================
# TRAINING LOOP: BERT-BASE-UNCASED FINE-TUNING
# End-to-end training: BERT backbone + custom classifier
# ============================================================

import json
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================
# OUTPUT FILES
# ============================================================

HISTORY_PATH = TASK_RESULTS_DIR / f"{RESULTS_PREFIX}_training_history.json"
FINAL_SUMMARY_PATH = TASK_RESULTS_DIR / f"{RESULTS_PREFIX}_training_summary.json"

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ============================================================
# FORWARD PASS
# ============================================================

def forward_batch(batch):
    input_ids, attention_mask, token_type_ids, labels = batch

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    token_type_ids = token_type_ids.to(device)
    labels = labels.to(device)

    outputs = bert_backbone(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )

    pooled_output = outputs.pooler_output
    logits = classifier(pooled_output)

    loss = criterion(logits, labels)
    preds = torch.argmax(logits, dim=1)

    return loss, preds, labels


# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(epoch):
    bert_backbone.train()
    classifier.train()

    loss_sum = 0.0
    total = 0
    y_true = []
    y_pred = []

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
        leave=True
    )

    for batch in progress_bar:
        optimizer.zero_grad(set_to_none=True)

        loss, preds, labels = forward_batch(batch)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        loss_sum += loss.item() * batch_size
        total += batch_size

        y_true.extend(labels.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.1e}"
        })

    return {
        "loss": float(loss_sum / total),
        "metrics": compute_metrics(y_true, y_pred),
    }


# ============================================================
# EVALUATE
# ============================================================

def evaluate(dataloader):
    bert_backbone.eval()
    classifier.eval()

    loss_sum = 0.0
    total = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in dataloader:
            loss, preds, labels = forward_batch(batch)

            batch_size = labels.size(0)

            loss_sum += loss.item() * batch_size
            total += batch_size

            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(preds.detach().cpu().numpy())

    return {
        "loss": float(loss_sum / total),
        "metrics": compute_metrics(y_true, y_pred),
    }


# ============================================================
# MAIN TRAINING LOOP
# ============================================================

training_history = []

best_val_f1 = -float("inf")
best_epoch = -1
early_stopping_counter = 0

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 80)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print("=" * 80)

    train_results = train_one_epoch(epoch)
    val_results = evaluate(val_loader)

    train_loss = train_results["loss"]
    val_loss = val_results["loss"]

    train_metrics = train_results["metrics"]
    val_metrics = val_results["metrics"]

    scheduler.step(val_loss)

    current_lr = float(optimizer.param_groups[0]["lr"])
    val_f1 = val_metrics["f1"]

    is_best = val_f1 > best_val_f1 + EARLY_STOPPING_MIN_DELTA

    if is_best:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        early_stopping_counter = 0

        torch.save(bert_backbone.state_dict(), BEST_BACKBONE_PATH)
        torch.save(classifier.state_dict(), BEST_CLASSIFIER_PATH)

    else:
        early_stopping_counter += 1

    epoch_record = {
        "epoch": epoch + 1,

        "train_loss": train_loss,
        "train_metrics": train_metrics,

        "val_loss": val_loss,
        "val_metrics": val_metrics,

        "learning_rate": current_lr,
        "is_best": is_best,
        "early_stopping_counter": early_stopping_counter,
    }

    training_history.append(epoch_record)

    with open(HISTORY_PATH, "w") as f:
        json.dump(training_history, f, indent=4)

    print(
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_f1={val_f1:.4f} | "
        f"lr={current_lr:.8f} | "
        f"best={is_best}"
    )

    if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break


# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

best_record = training_history[best_epoch - 1] if best_epoch > 0 else None
last_record = training_history[-1]

final_summary = {
    "task": TASK,
    "task_name": TASK_NAME,
    "model_id": MODEL_ID,

    "best_epoch": int(best_epoch),
    "best_val_f1": float(best_val_f1),
    "best_record": best_record,

    "last_epoch": int(last_record["epoch"]),
    "last_record": last_record,

    "max_len": int(MAX_LEN),
    "batch_size": int(BATCH_SIZE),
    "num_epochs_requested": int(NUM_EPOCHS),
    "num_epochs_completed": int(len(training_history)),

    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),

    "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
    "early_stopping_min_delta": float(EARLY_STOPPING_MIN_DELTA),

    "lr_factor": float(LR_FACTOR),
    "lr_patience": int(LR_PATIENCE),
    "lr_threshold": float(LR_THRESHOLD),
    "min_lr": float(MIN_LR),

    "bert_trainable_params": int(bert_trainable_params),
    "classifier_trainable_params": int(classifier_trainable_params),
    "trainable_params": int(trainable_params),

    "end_to_end_training": True,
    "backbone_frozen": False,

    "best_model_dir": str(BEST_MODEL_DIR),
}

with open(FINAL_SUMMARY_PATH, "w") as f:
    json.dump(final_summary, f, indent=4)

with open(BEST_TRAINING_INFO_PATH, "w") as f:
    json.dump(final_summary, f, indent=4)


# ============================================================
# FINAL PRINT ONLY
# ============================================================

print("\nTraining completed.")
print(f"Task: {TASK_NAME}")
print(f"Epochs completed: {len(training_history)}")
print(f"Best epoch: {best_epoch}")
print(f"Best validation F1: {best_val_f1:.4f}")

if best_record is not None:
    print(f"Best validation accuracy: {best_record['val_metrics']['accuracy']:.4f}")
    print(f"Best validation loss: {best_record['val_loss']:.4f}")

print(f"History saved in: {HISTORY_PATH}")
print(f"Summary saved in: {FINAL_SUMMARY_PATH}")
print(f"Best model saved in: {BEST_MODEL_DIR}")

In [ ]:
# ============================================================
# SAVE FINAL FINE-TUNED BERT EXPERT
# ============================================================

import json
import torch

# ------------------------------------------------------------
# Final output directory
# This is task-dependent:
# Models/sst_bert_base_uncased
# Models/mnli_bert_base_uncased
# ------------------------------------------------------------
FINAL_MODEL_DIR = TASK_MODEL_DIR
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load best checkpoint before saving final model
# ------------------------------------------------------------
bert_backbone.load_state_dict(torch.load(BEST_BACKBONE_PATH, map_location=device))
classifier.load_state_dict(torch.load(BEST_CLASSIFIER_PATH, map_location=device))

bert_backbone.eval()
classifier.eval()

# ------------------------------------------------------------
# Save fine-tuned BERT backbone
# ------------------------------------------------------------
bert_backbone.save_pretrained(FINAL_MODEL_DIR / "bert_backbone")

# ------------------------------------------------------------
# Save tokenizer
# ------------------------------------------------------------
tokenizer.save_pretrained(FINAL_MODEL_DIR / "tokenizer")

# ------------------------------------------------------------
# Save custom classifier
# ------------------------------------------------------------
torch.save(
    classifier.state_dict(),
    FINAL_MODEL_DIR / "classifier.pt"
)

# ------------------------------------------------------------
# Save complete checkpoint
# ------------------------------------------------------------
torch.save(
    {
        "task": TASK,
        "task_name": TASK_NAME,
        "model_id": MODEL_ID,
        "bert_backbone_state_dict": bert_backbone.state_dict(),
        "classifier_state_dict": classifier.state_dict(),
        "num_labels": NUM_LABELS,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
    },
    FINAL_MODEL_DIR / "full_checkpoint.pt"
)

# ------------------------------------------------------------
# Save model metadata
# ------------------------------------------------------------
model_metadata = {
    "model_type": "fine_tuned_bert_base_uncased_expert",
    "task": TASK,
    "task_name": TASK_NAME,
    "base_model": MODEL_ID,
    "num_labels": int(NUM_LABELS),
    "max_len": int(MAX_LEN),
    "batch_size": int(BATCH_SIZE),
    "num_epochs_requested": int(NUM_EPOCHS),
    "num_epochs_completed": int(len(training_history)),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "bert_trainable_params": int(bert_trainable_params),
    "classifier_trainable_params": int(classifier_trainable_params),
    "trainable_params": int(trainable_params),
    "best_epoch": int(best_epoch),
    "best_val_f1": float(best_val_f1),
    "end_to_end_training": True,
    "backbone_frozen": False,
}

with open(FINAL_MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=4)

print("Final fine-tuned BERT expert saved.")
print(f"Task: {TASK_NAME}")
print(f"Path: {FINAL_MODEL_DIR}")
print(f"Backbone: {FINAL_MODEL_DIR / 'bert_backbone'}")
print(f"Tokenizer: {FINAL_MODEL_DIR / 'tokenizer'}")
print(f"Classifier: {FINAL_MODEL_DIR / 'classifier.pt'}")

In [ ]:
# ============================================================
# PRINT TRAINING SUMMARY FROM JSON
# ============================================================

import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load training summary
# ------------------------------------------------------------
TRAINING_SUMMARY_PATH = TASK_RESULTS_DIR / f"{RESULTS_PREFIX}_training_summary.json"

with open(TRAINING_SUMMARY_PATH, "r") as f:
    training_summary = json.load(f)

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------
def print_best_metrics(training_summary):
    best_record = training_summary["best_record"]

    train_metrics = best_record["train_metrics"]
    val_metrics = best_record["val_metrics"]

    rows = [
        {
            "Split": "Train",
            "Loss": best_record["train_loss"],
            "Accuracy": train_metrics["accuracy"],
            "Precision": train_metrics["precision"],
            "Recall": train_metrics["recall"],
            "F1": train_metrics["f1"],
        },
        {
            "Split": "Validation",
            "Loss": best_record["val_loss"],
            "Accuracy": val_metrics["accuracy"],
            "Precision": val_metrics["precision"],
            "Recall": val_metrics["recall"],
            "F1": val_metrics["f1"],
        },
    ]

    df = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("BEST EPOCH METRICS")
    print("=" * 90)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def print_last_metrics(training_summary):
    last_record = training_summary["last_record"]

    train_metrics = last_record["train_metrics"]
    val_metrics = last_record["val_metrics"]

    rows = [
        {
            "Split": "Train",
            "Loss": last_record["train_loss"],
            "Accuracy": train_metrics["accuracy"],
            "Precision": train_metrics["precision"],
            "Recall": train_metrics["recall"],
            "F1": train_metrics["f1"],
        },
        {
            "Split": "Validation",
            "Loss": last_record["val_loss"],
            "Accuracy": val_metrics["accuracy"],
            "Precision": val_metrics["precision"],
            "Recall": val_metrics["recall"],
            "F1": val_metrics["f1"],
        },
    ]

    df = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("LAST EPOCH METRICS")
    print("=" * 90)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def print_experiment_info(training_summary):
    info_keys = [
        "task",
        "task_name",
        "model_id",
        "max_len",
        "batch_size",
        "num_epochs_requested",
        "num_epochs_completed",
        "learning_rate",
        "weight_decay",
        "trainable_params",
        "bert_trainable_params",
        "classifier_trainable_params",
        "best_epoch",
        "best_val_f1",
        "end_to_end_training",
        "backbone_frozen",
        "best_model_dir",
    ]

    rows = []

    for key in info_keys:
        if key in training_summary:
            rows.append({
                "Parameter": key,
                "Value": training_summary[key]
            })

    df_info = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("EXPERIMENT INFO")
    print("=" * 90)
    print(df_info.to_string(index=False))


# ------------------------------------------------------------
# Print everything
# ------------------------------------------------------------
print_best_metrics(training_summary)
print_last_metrics(training_summary)
print_experiment_info(training_summary)

print("\nTraining summary loaded from:")
print(TRAINING_SUMMARY_PATH)